In [ ]:
!pip install -q datasets bert-score textstat scispacy
!pip install -q git+https://github.com/feralvam/easse.git
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_lg-0.5.4.tar.gz
import os; os.kill(os.getpid(), 9)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.2/14.2 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 83.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.7/313.7 kB 32.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires n

In [ ]:
import os, json, numpy as np, spacy, textstat
from datasets import load_from_disk
from bert_score import score as compute_bertscore
from easse.sari import corpus_sari
from datetime import datetime
from google.colab import drive

drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/Gatekeepn't/Data/processed"
RESULTS_DIR = "/content/drive/MyDrive/Gatekeepn't/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

test_sells = load_from_disk(f"{ROOT}/test_sells")
test_medlane = load_from_disk(f"{ROOT}/test_medlane")
test_cochrane = load_from_disk(f"{ROOT}/test_cochrane")
test_plaba = load_from_disk(f"{ROOT}/test_plaba")

ent_nlp = spacy.load("en_core_sci_lg")

print(f"sells:    {len(test_sells)}")
print(f"medlane:  {len(test_medlane)}")
print(f"cochrane: {len(test_cochrane)}")
print(f"plaba:    {len(test_plaba)}")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


sells:    23416
medlane:  1010
cochrane: 480
plaba:    1000


In [ ]:
def extract_entities_batch(texts, batch_size=512):
    """
    run scispacy NER on a list of texts using nlp.pipe for speed.
    returns a list of sets, each containing lowercased entity strings.
    """
    all_ents = []
    for doc in ent_nlp.pipe(texts, batch_size=batch_size):
        all_ents.append(set(e.text.lower() for e in doc.ents))
    return all_ents


def entity_prf(src_ents_list, pred_ents_list):
    """
    precision = entities in prediction that were in source (hallucination check)
    recall = entities from source that survived in prediction (info loss check)
    """
    precs, recs, f1s = [], [], []
    for s_ents, p_ents in zip(src_ents_list, pred_ents_list):
        if len(p_ents) == 0:
            p = 1.0 if len(s_ents) == 0 else 0.0
        else:
            p = len(p_ents & s_ents) / len(p_ents)

        if len(s_ents) == 0:
            r = 1.0
        else:
            r = len(p_ents & s_ents) / len(s_ents)

        f = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        precs.append(p)
        recs.append(r)
        f1s.append(f)

    return np.mean(precs), np.mean(recs), np.mean(f1s)


def readability_deltas(sources, predictions):
    """
    fre delta positive = easier to read (good)
    fkg delta negative = lower grade level (good)
    """
    fre_d, fkg_d = [], []
    for src, pred in zip(sources, predictions):
        if src.strip() and pred.strip():
            fre_d.append(textstat.flesch_reading_ease(pred) - textstat.flesch_reading_ease(src))
            fkg_d.append(textstat.flesch_kincaid_grade(pred) - textstat.flesch_kincaid_grade(src))
    return np.mean(fre_d), np.mean(fkg_d)

In [ ]:
print("=" * 60)
print("PART 1: REFERENCE QUALITY AUDIT")
print("=" * 60)

ref_results = {}

for tag, ds in [("sells", test_sells), ("medlane", test_medlane),
                ("cochrane", test_cochrane), ("plaba", test_plaba)]:

    srcs = ds["source"]
    refs = ds["target"]

    print(f"\n--- {tag.upper()} ({len(ds)} examples) ---")

    # entity extraction on both source and reference
    print(f"  extracting entities from sources...")
    src_ents = extract_entities_batch(srcs)
    print(f"  extracting entities from references...")
    ref_ents = extract_entities_batch(refs)

    # did humans preserve medical terms?
    ep, er, ef = entity_prf(src_ents, ref_ents)

    # did humans actually make text easier?
    d_fre, d_fkg = readability_deltas(srcs, refs)

    ref_results[tag] = dict(
        n=len(ds),
        ref_ent_p=round(ep, 4),
        ref_ent_r=round(er, 4),
        ref_ent_f1=round(ef, 4),
        ref_d_fre=round(d_fre, 2),
        ref_d_fkg=round(d_fkg, 2),
    )

    print(f"  entity P:  {ep:.4f}")
    print(f"  entity R:  {er:.4f}")
    print(f"  entity F1: {ef:.4f}")
    print(f"  FRE delta: {d_fre:+.2f}")
    print(f"  FKG delta: {d_fkg:+.2f}")

PART 1: REFERENCE QUALITY AUDIT

--- SELLS (23416 examples) ---
  extracting entities from sources...
  extracting entities from references...
  entity P:  0.2142
  entity R:  0.1994
  entity F1: 0.1943
  FRE delta: +4.10
  FKG delta: -0.68

--- MEDLANE (1010 examples) ---
  extracting entities from sources...
  extracting entities from references...
  entity P:  0.6571
  entity R:  0.7036
  entity F1: 0.6743
  FRE delta: -10.07
  FKG delta: +2.17

--- COCHRANE (480 examples) ---
  extracting entities from sources...
  extracting entities from references...
  entity P:  0.4642
  entity R:  0.3138
  entity F1: 0.3651
  FRE delta: -3.63
  FKG delta: +2.08

--- PLABA (1000 examples) ---
  extracting entities from sources...
  extracting entities from references...
  entity P:  0.4765
  entity R:  0.4908
  entity F1: 0.4702
  FRE delta: +14.47
  FKG delta: -1.97


In [ ]:
from bert_score import BERTScorer

print("\n" + "=" * 60)
print("PART 2: COPY-SOURCE BASELINE")
print("=" * 60)

# set up BERTScorer once, fix tokenizer max_length to avoid overflow bug
scorer = BERTScorer(model_type="allenai/scibert_scivocab_uncased", batch_size=64)
scorer._tokenizer.model_max_length = 512

baseline_results = {}

for tag, ds in [("sells", test_sells), ("medlane", test_medlane),
                ("cochrane", test_cochrane), ("plaba", test_plaba)]:

    srcs = ds["source"]
    refs = ds["target"]

    print(f"\n--- {tag.upper()} ({len(ds)} examples) ---")

    # sari with source as prediction = no edits made
    print(f"  computing SARI...")
    copy_sari = corpus_sari(
        orig_sents=list(srcs),
        sys_sents=list(srcs),
        refs_sents=[list(refs)],
    )

    # bertscore between source and reference
    print(f"  computing BERTScore...")
    _, _, bs_f1 = scorer.score(list(srcs), list(refs))
    src_ref_bs = bs_f1.mean().item()

    baseline_results[tag] = dict(
        copy_sari=round(copy_sari, 2),
        src_ref_bertscore=round(src_ref_bs, 4),
    )

    print(f"  SARI (floor):        {copy_sari:.2f}")
    print(f"  BERTScore (src,ref): {src_ref_bs:.4f}")

In [ ]:
# merge both parts
combined = {}
for tag in ["sells", "medlane", "cochrane", "plaba"]:
    combined[tag] = {**ref_results[tag], **baseline_results[tag]}

# summary table
print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"{'dataset':<12} {'EntP':>7} {'EntR':>7} {'EntF1':>7} {'dFRE':>7} {'dFKG':>7} {'SARI':>7} {'BS':>7}")
print("-" * 70)
for tag in ["sells", "medlane", "cochrane", "plaba"]:
    r = combined[tag]
    print(f"{tag:<12} {r['ref_ent_p']:>7.4f} {r['ref_ent_r']:>7.4f} {r['ref_ent_f1']:>7.4f} "
          f"{r['ref_d_fre']:>+7.2f} {r['ref_d_fkg']:>+7.2f} {r['copy_sari']:>7.2f} {r['src_ref_bertscore']:>7.4f}")

# save
output = {
    "experiment": "exp0_baselines",
    "timestamp": datetime.now().isoformat(),
    "description": "reference quality audit + copy-source baseline on test sets",
    "results": combined,
}

save_path = f"{RESULTS_DIR}/exp0_baselines.json"
with open(save_path, "w") as f:
    json.dump(output, f, indent=2)
print(f"\nsaved to {save_path}")